In [4]:
# Menyiapkan Dataset

import numpy as np
import pandas as pd

np.random.seed(111)
n = 2500
menu_list = ["Kopi Susu", "Americano", "Cappuccino", "Roti Bakar", "Kentang Goreng", "Es Teh"]
cabang_list = ["Magelang", "Yogyakarta", "Semarang"]
waktu_list = ["Pagi", "Siang", "Sore", "Malam"]

data_tugas11 = pd.DataFrame({
    "nama_menu": np.random.choice(menu_list, size=n),
    "cabang": np.random.choice(cabang_list, size=n),
    "waktu_pesan": np.random.choice(waktu_list, size=n),
    "qty": np.random.randint(1, 6, size=n),
    "harga_satuan": np.random.choice([15000, 18000, 20000, 22000, 25000, 30000], size=n),
})
data_tugas11["total_penjualan"] = data_tugas11["qty"] * data_tugas11["harga_satuan"] + np.random.normal(0, 8000, size=n)
data_tugas11.to_csv("tugas11_penjualan.csv", index=False)

print(f"Dataset berhasil dibuat: {len(data_tugas11)} baris")
print(data_tugas11.head())

# Menginisialisasi SparkSession
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("Tugas11-KopiNusantara") \
    .master("local[*]") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")
spark.conf.set("spark.sql.shuffle.partitions", "4")

df_tugas = spark.read.csv("tugas11_penjualan.csv", header=True, inferSchema=True)

print("SparkSession siap. Versi Spark:", spark.version)
df_tugas.show(5)
print("Total baris data:", df_tugas.count())


Dataset berhasil dibuat: 2500 baris
        nama_menu      cabang waktu_pesan  qty  harga_satuan  total_penjualan
0  Kentang Goreng    Semarang       Malam    1         30000     27750.163532
1  Kentang Goreng  Yogyakarta        Sore    3         20000     64216.160744
2  Kentang Goreng    Semarang       Siang    5         22000    108722.792592
3  Kentang Goreng  Yogyakarta        Pagi    2         20000     54483.205872
4      Roti Bakar    Magelang       Malam    2         30000     58157.944120
SparkSession siap. Versi Spark: 3.5.9
+--------------+----------+-----------+---+------------+------------------+
|     nama_menu|    cabang|waktu_pesan|qty|harga_satuan|   total_penjualan|
+--------------+----------+-----------+---+------------+------------------+
|Kentang Goreng|  Semarang|      Malam|  1|       30000|27750.163531558544|
|Kentang Goreng|Yogyakarta|       Sore|  3|       20000| 64216.16074371864|
|Kentang Goreng|  Semarang|      Siang|  5|       22000|108722.79259154423|
|K

In [2]:
# A Feature Engineering

from pyspark.ml.feature import StringIndexer, VectorAssembler

# 1. Mendefinisikan StringIndexer untuk 3 kolom teks
indexer_menu = StringIndexer(inputCol="nama_menu", outputCol="menu_idx")
indexer_cabang = StringIndexer(inputCol="cabang", outputCol="cabang_idx")
indexer_waktu = StringIndexer(inputCol="waktu_pesan", outputCol="waktu_idx")

# 2. Melatih (fit) dan mengonversi (transform) kolom teks menjadi angka
model_idx_menu = indexer_menu.fit(df_tugas)
df_tugas = model_idx_menu.transform(df_tugas)

model_idx_cabang = indexer_cabang.fit(df_tugas)
df_tugas = model_idx_cabang.transform(df_tugas)

model_idx_waktu = indexer_waktu.fit(df_tugas)
df_tugas = model_idx_waktu.transform(df_tugas)

# 3. Merangkai seluruh 5 fitur masukan menjadi 1 kolom vektor "features"
assembler_tugas = VectorAssembler(
    inputCols=["qty", "harga_satuan", "menu_idx", "cabang_idx", "waktu_idx"],
    outputCol="features"
)
df_tugas_final = assembler_tugas.transform(df_tugas)

df_tugas_final.select("features", "total_penjualan").show(5, truncate=False)

+-------------------------+------------------+
|features                 |total_penjualan   |
+-------------------------+------------------+
|[1.0,30000.0,5.0,2.0,1.0]|27750.163531558544|
|[3.0,20000.0,5.0,1.0,3.0]|64216.16074371864 |
|[5.0,22000.0,5.0,2.0,2.0]|108722.79259154423|
|[2.0,20000.0,5.0,1.0,0.0]|54483.205871792496|
|[2.0,30000.0,4.0,0.0,1.0]|58157.94412015849 |
+-------------------------+------------------+
only showing top 5 rows



In [3]:
# B. Train/Test Split & Pelatihan Model
from pyspark.ml.regression import LinearRegression

# 1. Membagi data (75% training, 25% testing)
train_tugas, test_tugas = df_tugas_final.randomSplit([0.75, 0.25], seed=101)

# 2. Melatih model LinearRegression
lr_tugas = LinearRegression(featuresCol="features", labelCol="total_penjualan")
model_tugas = lr_tugas.fit(train_tugas)

print(f"Jumlah data training : {train_tugas.count()}")
print(f"Jumlah data testing  : {test_tugas.count()}")
print("Koefisien tiap fitur  :", model_tugas.coefficients)
print("Intercept            :", model_tugas.intercept)


Jumlah data training : 1839
Jumlah data testing  : 661
Koefisien tiap fitur  : [22046.663271025758,3.022920868160672,-123.21425598657399,279.28452461677676,339.6387953231708]
Intercept            : -66903.17766485889


In [5]:
# C. Evaluasi Model
from pyspark.ml.evaluation import RegressionEvaluator

# 1. Membuat prediksi pada data testing
prediksi_tugas = model_tugas.transform(test_tugas)

# 2. Menghitung RMSE
evaluator_rmse = RegressionEvaluator(labelCol="total_penjualan", predictionCol="prediction", metricName="rmse")
rmse_tugas = evaluator_rmse.evaluate(prediksi_tugas)

# 3. Menghitung R²
evaluator_r2 = RegressionEvaluator(labelCol="total_penjualan", predictionCol="prediction", metricName="r2")
r2_tugas = evaluator_r2.evaluate(prediksi_tugas)

print(f"Hasil Evaluasi Model Utama:")
print(f"RMSE : Rp {rmse_tugas:,.0f}")
print(f"R²   : {r2_tugas:.4f} ({r2_tugas*100:.1f}% variasi data berhasil dijelaskan)")


Hasil Evaluasi Model Utama:
RMSE : Rp 10,901
R²   : 0.9107 (91.1% variasi data berhasil dijelaskan)


Nilai R² sebesar 0,9107 menunjukkan bahwa model sudah cukup baik dalam menangkap pola penjualan. Rata-rata kesalahan prediksi (RMSE) sekitar Rp 10.901, atau sekitar 17% dari rata-rata total_penjualan yang sekitar Rp 65.000. Jadi model layak dipakai sebagai gambaran omzet untuk perencanaan stok bahan baku, tetapi belum cukup presisi bila dipakai untuk memprediksi satu transaksi secara tepat.

In [6]:
# D. Eksperimen Perbandingan Fitur
# 1. Merangkai VectorAssembler kedua HANYA untuk "qty" dan "harga_satuan"
assembler_eksperimen = VectorAssembler(
    inputCols=["qty", "harga_satuan"],
    outputCol="features_eksperimen"
)
df_eksperimen = assembler_eksperimen.transform(df_tugas_final)

# 2. Split data 75:25 dengan seed=101 yang sama
train_eks, test_eks = df_eksperimen.randomSplit([0.75, 0.25], seed=101)

# 3. Melatih model kedua
lr_eks = LinearRegression(featuresCol="features_eksperimen", labelCol="total_penjualan")
model_eks = lr_eks.fit(train_eks)

# 4. Evaluasi R² model kedua
prediksi_eks = model_eks.transform(test_eks)
r2_eks = evaluator_r2.evaluate(prediksi_eks)

print(f"R² Model Utama (5 Fitur)            : {r2_tugas:.4f}")
print(f"R² Model Eksperimen (Qty & Harga)  : {r2_eks:.4f}")


R² Model Utama (5 Fitur)            : 0.9107
R² Model Eksperimen (Qty & Harga)  : 0.9108


Nilai R² kedua model hampir identik, sehingga menambahkan informasi menu, cabang, dan waktu pesan tidak meningkatkan akurasi secara berarti. Hal ini terjadi karena total_penjualan pada dataset ini dibentuk dari qty × harga_satuan ditambah noise acak, sehingga ketiga kolom tambahan tidak membawa informasi baru.

In [8]:
# E. Prediksi Skenario Baru

import pandas as pd

# 1. Membuat DataFrame 3 skenario pesanan baru (data dummy)
data_baru = pd.DataFrame({
    "nama_menu": ["Roti Bakar", "Kopi Susu", "Americano"],
    "cabang": ["Semarang", "Magelang", "Yogyakarta"],
    "waktu_pesan": ["Pagi", "Siang", "Malam"],
    "qty": [2, 4, 1],
    "harga_satuan": [22000, 18000, 15000]
})
df_baru = spark.createDataFrame(data_baru)

# 2. Mengubah kategori menggunakan objek StringIndexer yang SAMA dari Bagian A
df_baru = model_idx_menu.transform(df_baru)
df_baru = model_idx_cabang.transform(df_baru)
df_baru = model_idx_waktu.transform(df_baru)

# 3. Merangkai fitur baru
df_baru_final = assembler_tugas.transform(df_baru)

# 4. Melakukan prediksi total_penjualan
prediksi_skenario = model_tugas.transform(df_baru_final)

print("Hasil Prediksi 3 Skenario Pesanan Baru:")
prediksi_skenario.select(
    "nama_menu", "cabang", "waktu_pesan", "qty", "harga_satuan", "prediction"
).show()


Hasil Prediksi 3 Skenario Pesanan Baru:
+----------+----------+-----------+---+------------+------------------+
| nama_menu|    cabang|waktu_pesan|qty|harga_satuan|        prediction|
+----------+----------+-----------+---+------------+------------------+
|Roti Bakar|  Semarang|       Pagi|  2|       22000| 43760.12000201468|
| Kopi Susu|  Magelang|      Siang|  4|       18000| 76252.11438079602|
| Americano|Yogyakarta|      Malam|  1|       15000|1106.2219485168898|
+----------+----------+-----------+---+------------+------------------+

